In [1]:
#all necessary imports
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# TA indicators
import ta
from ta.trend import PSARIndicator

# Data Normalization and Scaling
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scikit-learn utilities
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# hyperparameter optimization
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

print("Torch version:", torch.__version__)
print("Polars version:", pl.__version__)
print("TQDM imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("TA library imported successfully!")
print("Optuna version:",optuna.__version__)

Training device: cuda
Torch version: 2.5.1+cu121
Polars version: 0.20.26
TQDM imported successfully!
NumPy version: 1.26.4
Pandas version: 2.2.2
TA library imported successfully!
Optuna version: 4.8.0


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Data preprocessing function
# change this to the desired stock ticker
company = "8180"
files = [ f"{company.lower()}_daily.csv"]
dfs = [pl.read_csv(f, try_parse_dates=True) for f in files]

combined_df = pl.concat(dfs)

combined_df = combined_df.sort(["ticker", "datetime"])

print(combined_df)

AAPL: (9964, 33) | 1986-08-22 21:30:00 → 2026-03-19 21:30:00
NVDA: (6795, 33) | 1999-03-16 22:30:00 → 2026-03-19 21:30:00


In [ ]:
# Convert to pandas for easier manipulation
pandas_df = combined_df.to_pandas()
print(pandas_df)

# PSAR Indicator + Trend Labeling
processed_stocks = []

for ticker in pandas_df["ticker"].unique():
    stock = pandas_df[pandas_df["ticker"] == ticker].copy()
    stock = stock.sort_values("datetime")

    psar_indicator = PSARIndicator(
        high=stock["high_price"],
        low=stock["low_price"],
        close=stock["close_price"],
        step=0.02,
        max_step=0.2
    )

    stock["psar"] = psar_indicator.psar()

    # Binary classification target (Trend Definition)
    stock["trend"] = (stock["close_price"] > stock["psar"]).astype(int)

    processed_stocks.append(stock)

df = pd.concat(processed_stocks).sort_values(["ticker", "datetime"]).reset_index(drop=True)

print(df.head())


In [ ]:
# removing rows with NaN values
df = df.dropna().reset_index(drop=True)
print(df)

In [ ]:
train_list = []
test_list  = []

for ticker in df["ticker"].unique():
    ticker_df  = df[df["ticker"] == ticker].sort_values("datetime").reset_index(drop=True)
    split_idx  = int(len(ticker_df) * 0.8)
    train_list.append(ticker_df.iloc[:split_idx])
    test_list.append(ticker_df.iloc[split_idx:])

train_df = pd.concat(train_list).sort_values(["ticker", "datetime"]).reset_index(drop=True)
test_df  = pd.concat(test_list).sort_values(["ticker", "datetime"]).reset_index(drop=True)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows : {len(test_df):,}")
print(f"\nTrain/Test split per ticker:")
for ticker in df["ticker"].unique():
    tr = len(train_df[train_df["ticker"] == ticker])
    te = len(test_df[test_df["ticker"]  == ticker])
    tr_end = train_df[train_df["ticker"] == ticker]["datetime"].max()
    te_start = test_df[test_df["ticker"] == ticker]["datetime"].min()
    print(f"  {ticker}: train={tr:,} (up to {tr_end.date()}) | "
          f"test={te:,} (from {te_start.date()})")

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
import numpy as np

features    = ["open_price", "high_price", "low_price", "close_price", "volume"]
TICKERS     = [8180]
WINDOW_SIZE = 30
HORIZON     = 4
SCALE_CLIP  = 5.0
scalers     = {}

X_train_list, y_train_list = [], []
X_test_list,  y_test_list  = [], []

def compute_log_returns(df, features, clip_val=0.5):
    df = df.copy()
    price_cols = ["open_price", "high_price", "low_price", "close_price"]

    for col in price_cols:
        df[col] = np.log(df[col] / df[col].shift(1))
        df[col] = df[col].clip(-clip_val, clip_val)

    df["volume"] = np.log(df["volume"] / df["volume"].shift(1))
    df["volume"] = df["volume"].clip(-clip_val, clip_val)

    df = df.dropna().reset_index(drop=True)
    return df

for ticker in TICKERS:
    print(f"\nProcessing {ticker}...")

    train_tk = train_df[train_df["ticker"] == ticker].sort_values("datetime").reset_index(drop=True)
    test_tk  = test_df[test_df["ticker"]   == ticker].sort_values("datetime").reset_index(drop=True)

    if len(train_tk) == 0 or len(test_tk) == 0:
        print(f"  WARNING — no data for {ticker}, skipping.")
        continue

    train_tk = compute_log_returns(train_tk, features)
    test_tk  = compute_log_returns(test_tk,  features)

    sc = StandardScaler()
    train_feat_scaled = sc.fit_transform(train_tk[features].values)
    test_feat_scaled  = sc.transform(test_tk[features].values)

    train_feat_scaled = np.clip(train_feat_scaled, -SCALE_CLIP, SCALE_CLIP)
    test_feat_scaled  = np.clip(test_feat_scaled,  -SCALE_CLIP, SCALE_CLIP)
    scalers[ticker]   = sc

    print(f"  Train: {len(train_tk):,} rows | "
          f"scaled min/max: {train_feat_scaled.min():.3f} / {train_feat_scaled.max():.3f}")
    print(f"  Test : {len(test_tk):,} rows  | "
          f"scaled min/max: {test_feat_scaled.min():.3f} / {test_feat_scaled.max():.3f}")

    train_trend = train_tk["trend"].values
    test_trend  = test_tk["trend"].values

    for i in range(WINDOW_SIZE, len(train_feat_scaled) - HORIZON):
        X_train_list.append(train_feat_scaled[i - WINDOW_SIZE:i])
        y_train_list.append(train_trend[i:i + HORIZON])

    for i in range(WINDOW_SIZE, len(test_feat_scaled) - HORIZON):
        X_test_list.append(test_feat_scaled[i - WINDOW_SIZE:i])
        y_test_list.append(test_trend[i:i + HORIZON])

# ── Stack ─────────────────────────────────────────────────────────────────────
X_train = np.array(X_train_list)   # (n, 30, 5)
y_train = np.array(y_train_list)   # (n, 4)
X_test  = np.array(X_test_list)    # (n, 30, 5)
y_test  = np.array(y_test_list)    # (n, 4)

joblib.dump(scalers, "scalers_per_ticker.pkl")

# ── Verification ──────────────────────────────────────────────────────────────
print("\n── Final Shapes ──────────────────────────────────────────")
print(f"X_train: {X_train.shape}  — (n, {WINDOW_SIZE}, {len(features)})")
print(f"y_train: {y_train.shape}  — (n, {HORIZON} horizons)")
print(f"X_test : {X_test.shape}")
print(f"y_test : {y_test.shape}")
print(f"\nX_train min/max: {X_train.min():.4f} / {X_train.max():.4f}")
print(f"X_test  min/max: {X_test.min():.4f}  / {X_test.max():.4f}")
print(f"\ny_train unique: {np.unique(y_train)}")
print(f"y_test  unique: {np.unique(y_test)}")
print(f"\nTotal train sequences: {len(X_train):,}")
print(f"Total test  sequences: {len(X_test):,}")

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=30, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)

        # Build positional encoding matrix (max_len, d_model)
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()        # (30, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)   # even indices
        pe[:, 1::2] = torch.cos(position * div_term)   # odd indices
        pe = pe.unsqueeze(0)                            # (1, 30, d_model)

        # Register as buffer — not a parameter, but saved with model
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# v2
class StockTrendTransformer(nn.Module):
    def __init__(self, input_size=5, d_model=64, nhead=8, num_layers=3,
                 dim_feedforward=256, horizon=4, dropout=0.2,
                 max_len=30, pooling="last"):   # ← add pooling param
        super(StockTrendTransformer, self).__init__()
        self.d_model  = d_model
        self.horizon  = horizon
        self.pooling  = pooling               # ← store it

        self.input_proj = nn.Linear(input_size, d_model)

        # CLS token needed only if pooling="cls"
        if pooling == "cls":
            self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
            self.pos_encoder = PositionalEncoding(d_model, max_len + 1, dropout)
        else:
            self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = nhead,
            dim_feedforward = dim_feedforward,
            dropout         = dropout,
            batch_first     = True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )

        self.bn      = nn.BatchNorm1d(d_model)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(d_model, horizon)

    def forward(self, x):
        out = self.input_proj(x)              # (batch, 30, d_model)

        # Prepend CLS token if needed
        if self.pooling == "cls":
            cls = self.cls_token.expand(x.size(0), -1, -1)  # (batch, 1, d_model)
            out = torch.cat([cls, out], dim=1)               # (batch, 31, d_model)

        out = self.pos_encoder(out)
        out = self.transformer_encoder(out)   # (batch, 30or31, d_model)

        # ── Pooling strategy ──────────────────────────────────────────────────
        if self.pooling == "last":
            out = out[:, -1, :]               # last timestep
        elif self.pooling == "mean":
            out = out.mean(dim=1)             # average all timesteps
        elif self.pooling == "cls":
            out = out[:, 0, :]               # CLS token at position 0

        out = self.bn(out)
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [ ]:
# Verify output shape before training
model_test = StockTrendTransformer().to("cpu")
dummy      = torch.zeros(8, 30, 5)
output     = model_test(dummy)
print("Output shape:", output.shape)   # must be (8, 4)

total_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


In [ ]:
# training loop setup starts here
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS      = 100 # reduced for quick testing; increase to for real training
BATCH_SIZE  = 32
LR          = 1e-3
PATIENCE    = 15          # early stopping patience
HORIZON     = 4           # t+1 … t+4

print(f"Training on: {DEVICE}")

In [ ]:
# DATASET & DATALOADER
X_train_t = torch.tensor(X_train, dtype=torch.float32)  # was X_train_scaled
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)  # was X_test_scaled
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t,  y_test_t)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

X_train_t = torch.tensor(X_train, dtype=torch.float32)  # was X_train_scaled
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)  # was X_test_scaled
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t,  y_test_t)

In [ ]:
def objective(trial):

    # ── Hyperparameters ───────────────────────────────────────────────────────
    model_config    = trial.suggest_categorical("model_config", [
        "32_2", "32_4",
        "64_2", "64_4", "64_8",
        "96_4", "96_8",
        "128_4", "128_8"
    ])
    d_model, nhead  = [int(x) for x in model_config.split("_")]

    ff_multiplier   = trial.suggest_categorical("ff_multiplier",     [2, 4])
    dim_feedforward = d_model * ff_multiplier
    num_layers      = trial.suggest_int("num_layers",                 1, 4)
    dropout         = trial.suggest_float("dropout",                  0.1, 0.4, step=0.1)
    lr              = trial.suggest_float("lr",                       1e-4, 1e-2, log=True)
    weight_decay    = trial.suggest_float("weight_decay",             1e-5, 1e-2, log=True)
    batch_size      = trial.suggest_categorical("batch_size",         [32, 64])
    factor          = trial.suggest_float("factor",                   0.3, 0.8, step=0.1)
    patience_sch    = trial.suggest_int("scheduler_patience",         3, 7)
    pooling         = trial.suggest_categorical("pooling",            ["last", "mean", "cls"])
    # ↑ separate suggest_categorical call — not merged into model_config

    # ── Loaders ───────────────────────────────────────────────────────────────
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

    # ── Build model with pooling param ────────────────────────────────────────
    model = StockTrendTransformer(
        input_size      = 5,
        d_model         = d_model,
        nhead           = nhead,
        num_layers      = num_layers,
        dim_feedforward = dim_feedforward,
        horizon         = 4,
        dropout         = dropout,
        max_len         = 30,
        pooling         = pooling    # ← pass pooling to model
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=patience_sch, factor=factor
    )

    OPTUNA_EPOCHS   = 30
    OPTUNA_PATIENCE = 7
    best_val_loss   = float("inf")
    patience_count  = 0

    for epoch in range(1, OPTUNA_EPOCHS + 1):

        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                val_loss += criterion(model(X_batch), y_batch).item()
        val_loss /= len(test_loader)

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= OPTUNA_PATIENCE:
            break

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_loss

In [ ]:
# Optimization here
study = optuna.create_study(
    direction = "minimize",
    pruner    = optuna.pruners.MedianPruner(
        n_startup_trials = 5,
        n_warmup_steps   = 5
    )
)

print("Starting Optuna search for Transformer — 50 trials...")
study.optimize(objective, n_trials=50, show_progress_bar=True)

# ── Results
best = study.best_params
d_model, nhead = [int(x) for x in best["model_config"].split("_")]

print("\n" + "=" * 50)
print("OPTUNA SEARCH COMPLETE")
print("=" * 50)
print(f"Best val_loss   : {study.best_value:.4f}")
print(f"d_model         : {d_model}")
print(f"nhead           : {nhead}")
print(f"dim_feedforward : {d_model * best['ff_multiplier']}")
for k, v in best.items():
    if k != "model_config":
        print(f"  {k:<22}: {v}")

Training starts here

In [ ]:
best = study.best_params
d_model, nhead = [int(x) for x in best["model_config"].split("_")]
dim_feedforward = d_model * best["ff_multiplier"]

print("Training Transformer with Optuna best params:")
print(f"  {'d_model':<22}: {d_model}")
print(f"  {'nhead':<22}: {nhead}")
print(f"  {'dim_feedforward':<22}: {dim_feedforward}")
for k, v in best.items():
    if k not in ["model_config", "ff_multiplier"]:
        print(f"  {k:<22}: {v}")

In [ ]:

d_model, nhead  = [int(x) for x in best["model_config"].split("_")]
dim_feedforward = d_model * best["ff_multiplier"]

print("Best Optuna params:")
print(f"  d_model         : {d_model}")
print(f"  nhead           : {nhead}")
print(f"  dim_feedforward : {dim_feedforward}")
for k, v in best.items():
    if k not in ["model_config", "ff_multiplier"]:
        print(f"  {k:<22}: {v}")

# ── Build model with best params ──────────────────────────────────────────────
model = StockTrendTransformer(
    input_size      = 5,
    d_model         = d_model,
    nhead           = nhead,
    num_layers      = best["num_layers"],
    dim_feedforward = dim_feedforward,
    horizon         = HORIZON,
    dropout         = best["dropout"],
    max_len         = WINDOW_SIZE,
    pooling         = best["pooling"]
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr           = best["lr"],
    weight_decay = best["weight_decay"]
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min",
    patience = best["scheduler_patience"],
    factor   = best["factor"]
)

# ── Verify ────────────────────────────────────────────────────────────────────
print(f"\nTraining on     : {DEVICE}")
print(f"Initial LR      : {optimizer.param_groups[0]['lr']:.6f}")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters      : {total_params:,}")
print(f"d_model         : {model.d_model}")
print(f"nhead           : {nhead}")
print(f"num_layers      : {best['num_layers']}")
print(f"dim_feedforward : {dim_feedforward}")
print(f"pooling         : {model.pooling}")
print(f"dropout         : {best['dropout']}")
print(f"horizon         : {model.horizon}")

In [ ]:
#  3. HELPER: EVALUATE ON ANY LOADER
def evaluate(model, loader, criterion, device):
    """
    Returns avg loss, per-horizon accuracy, per-horizon F1.
    model.eval() + torch.no_grad() ensures no gradient computation.
    """
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)                       # raw logits (batch, 4)
            loss   = criterion(logits, y_batch)
            total_loss += loss.item()

            probs  = torch.sigmoid(logits)                # convert to probabilities
            preds  = (probs > 0.5).float()                # threshold → 0 or 1

            all_preds.append(preds.cpu().numpy())
            all_labels.append(y_batch.cpu().numpy())

    all_preds  = np.concatenate(all_preds,  axis=0)      # (N, 4)
    all_labels = np.concatenate(all_labels, axis=0)      # (N, 4)

    # Per-horizon metrics (column = t+1, t+2, t+3, t+4)
    accs = [accuracy_score(all_labels[:, h], all_preds[:, h]) for h in range(HORIZON)]
    f1s  = [f1_score(all_labels[:, h], all_preds[:, h], zero_division=0) for h in range(HORIZON)]

    avg_loss = total_loss / len(loader)
    return avg_loss, accs, f1s


In [ ]:
# Training Loop with early stopping
history = {
    "train_loss": [], "val_loss": [],
    "val_acc": [],    "val_f1":  []
}

best_val_loss  = float("inf")
patience_count = 0
best_weights   = None

for epoch in range(1, EPOCHS + 1):

    # ── TRAIN PHASE ──────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()              # clear gradients from last step

        logits = model(X_batch)            # forward pass
        loss   = criterion(logits, y_batch)

        loss.backward()                    # backpropagation

        # Gradient clipping — prevents exploding gradients (critical for LSTMs)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()                   # update weights
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ── VALIDATION PHASE ─────────────────────────────────────────────────────
    val_loss, val_accs, val_f1s = evaluate(model, test_loader, criterion, DEVICE)

    # Step scheduler based on val_loss
    scheduler.step(val_loss)

    # Log history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(np.mean(val_accs))    # mean across horizons
    history["val_f1"].append(np.mean(val_f1s))

    # ── EARLY STOPPING ───────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_weights   = {k: v.clone() for k, v in model.state_dict().items()}
        patience_count = 0
        tag = "best"
    else:
        patience_count += 1
        tag = f" {patience_count}/{PATIENCE}"


    current_lr = optimizer.param_groups[0]["lr"]

    # Print every 10 epochs
    # if epoch % 10 == 0 or epoch == 1:
    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {np.mean(val_accs):.4f} | "
        f"Val F1: {np.mean(val_f1s):.4f} | "
        f"LR: {current_lr:.6f} | "
        f"{tag}"
    )

    if patience_count >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        print(f"Best val_loss was at epoch {epoch - PATIENCE}: {best_val_loss:.4f}")
        break

# Restore best weights
if best_weights is not None:
    model.load_state_dict(best_weights)
    print(f"\nRestored best model (val_loss={best_val_loss:.4f})")
else:
    print("\nWarning: No best weights saved — using final weights")

In [ ]:
#  5. FINAL EVALUATION
_, final_accs, final_f1s = evaluate(model, test_loader, criterion, DEVICE)

lines = []
lines.append("\nPer-Horizon Test Results:")
lines.append(f"{'Horizon':<12} {'Accuracy':>10} {'F1 Score':>10}")
lines.append("-" * 34)

for h in range(HORIZON):
    lines.append(f"  t+{h+1:<8}  {final_accs[h]:>10.4f} {final_f1s[h]:>10.4f}")

lines.append("-" * 34)
lines.append(f"  {'Mean':<10}  {np.mean(final_accs):>10.4f} {np.mean(final_f1s):>10.4f}")

# Join into one string
output_text = "\n".join(lines)

# Print to console
print(output_text)

# Save to file
with open(f'evaluation.txt({company})', "w") as f:
    f.write(output_text)

In [ ]:
# ── 6. PLOT TRAINING CURVES
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'Transformer Training History({company})', fontsize=14, fontweight="bold")

axes[0].plot(history["train_loss"], label="Train Loss", color="steelblue")
axes[0].plot(history["val_loss"],   label="Val Loss",   color="tomato")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(history["val_acc"], color="seagreen")
axes[1].set_title("Val Accuracy (Mean)"); axes[1].set_xlabel("Epoch")

axes[2].plot(history["val_f1"], color="darkorchid")
axes[2].set_title("Val F1 Score (Mean)"); axes[2].set_xlabel("Epoch")

plt.tight_layout()
plt.savefig(f'Transformer_training_history({company}).png', dpi=150)
plt.show()

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state":  optimizer.state_dict(),
    "val_loss":         best_val_loss,
    "history":          history,
    "config": {
        "input_size":       5,
        "d_model":          d_model,
        "nhead":            nhead,
        "num_layers":       best["num_layers"],
        "dim_feedforward":  dim_feedforward,
        "horizon":          HORIZON,
        "dropout":          best["dropout"],
        "window_size":      WINDOW_SIZE,
        "max_len":          WINDOW_SIZE,
        "pooling":          best["pooling"],
        "lr":               best["lr"],
        "weight_decay":     best["weight_decay"],
        "batch_size":       best["batch_size"],
        "factor":           best["factor"],
        "scheduler_patience": best["scheduler_patience"],
    }
}, f'Transformer_best({company}).pth')
print(f'Model saved to Transformer_best({company}).pth')

# ── Verify saved config ───────────────────────────────────────────────────────
ck = torch.load(f'Transformer_best({company}).pth', map_location="cpu", weights_only=False)
print("\nSaved config:")
for k, v in ck["config"].items():
    print(f"  {k:<22}: {v}")
print(f"  {'val_loss':<22}: {ck['val_loss']:.4f}")